# Qwen3-Embedding-4B — thử nghiệm riêng trên Kaggle

Dùng model có sẵn, **không huấn luyện/fine-tune**. Cùng 40 chunks, 90 query nháp và K=3 với BGE-M3; nhãn và nội dung giữ nguyên. Model Qwen xuất vector 2560 chiều, có instruction chỉ ở query, last-token pooling và chuẩn hóa L2.

1. Upload **`pilot-retrieval-qwen3-4b-input.zip`** thành Dataset **Private**, rồi Add Input vào notebook này. Đây là gói riêng, không dùng ZIP đầu vào của BGE.
2. Bật Internet và GPU trong Settings (T4 nếu có). Bản sửa OOM v2 dùng Transformers trực tiếp, FP16 và chia trọng số qua hai GPU khi có T4 ×2. Batch mặc định 2. Dừng phiên Kaggle cũ rồi bắt đầu session mới để giải phóng model/tensor còn giữ; chỉ chạy lại ô lỗi không làm sạch VRAM. Không cần FlashAttention hay API key.
3. Chọn **Save Version → Save & Run All**, chờ phiên nền bắt đầu trước khi tắt máy.
4. Khi hoàn thành, tải **`pilot-retrieval-qwen3-4b.zip`** ở Output. ZIP/khối index của BGE-M3 không bị ghi đè.

Gói chứa baseline BGE-M3 đã chạy để in bảng so sánh. Các query là bộ phát triển tự biên soạn, chưa là test độc lập hoặc bằng chứng hiệu quả học tập. Tài nguyên/quota Kaggle vẫn áp dụng; tải model khoảng 8 GB có thể chiếm phần lớn thời gian.

**Quan trọng:** cập nhật cả notebook này và Dataset input ZIP bản v2. Không gắn đồng thời nhiều bản ZIP Qwen. Giữ nguyên các kết quả BGE.


In [1]:
from pathlib import Path
from zipfile import ZipFile
import hashlib, json, sys, uuid

INPUT_PATH = None  # Nếu gắn nhiều gói Qwen, điền đường dẫn ZIP/thư mục cụ thể.
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
CACHE_ROOT = Path('/kaggle/temp')
if not INPUT_ROOT.is_dir():
    raise RuntimeError('Notebook này dành riêng cho Kaggle. Import notebook và Add Input trước.')
INPUT_NAME = 'pilot-retrieval-qwen3-4b-input.zip'
PREFIX = 'pilot-retrieval-qwen3-4b/'
INPUT_FILES = ['backend/app/__init__.py', 'backend/app/contracts.py', 'backend/app/curriculum.py', 'backend/app/pilot_contracts.py', 'backend/app/retrieval_contracts.py', 'backend/app/retrieval.py', 'backend/app/local_embedding.py', 'research/retrieval/evaluate.py', 'research/retrieval/requirements.txt', 'research/retrieval/qwen3_4b/experiment.py', 'research/retrieval/qwen3_4b/requirements.txt', 'corpus.json', 'queries.json', 'bge-baseline.json', 'manifest.json']
EXPECTED_ENCODER = {'model_id': 'Qwen/Qwen3-Embedding-4B', 'revision': '5cf2132abc99cad020ac570b19d031efec650f2b', 'dimensions': 2560, 'max_length': 512, 'pooling': 'last_token', 'normalize': True, 'query_prefix': 'Instruct: Given a Vietnamese math learning question, retrieve relevant explanations or hints addressing the mathematical concept and the requested level of support.\nQuery:', 'passage_prefix': ''}
MAX_INPUT_BYTES = 5*1024*1024

def find_input(root):
    archives = sorted(root.rglob(INPUT_NAME))
    candidates = list(archives)
    if not candidates:
        for manifest_path in sorted(root.rglob('manifest.json')):
            if manifest_path.stat().st_size > 65536:
                continue
            try:
                value = json.loads(manifest_path.read_text(encoding='utf-8'))
            except (ValueError, UnicodeError):
                continue
            if value.get('experiment') == 'qwen3-4b-v1':
                candidates.append(manifest_path.parent)
    if not candidates:
        raise ValueError('Thiếu input Qwen. Add Input Dataset chứa '+INPUT_NAME)
    if len(candidates) != 1:
        raise ValueError('Có nhiều input Qwen; điền INPUT_PATH: '+str(candidates))
    return candidates[0]

source = Path(INPUT_PATH) if INPUT_PATH else find_input(INPUT_ROOT)
if source.is_file():
    if source.stat().st_size > MAX_INPUT_BYTES:
        raise ValueError('Input ZIP quá lớn.')
    with ZipFile(source) as bundle:
        expected = {PREFIX+name for name in INPUT_FILES}
        if set(bundle.namelist()) != expected or len(bundle.namelist()) != len(expected):
            raise ValueError('ZIP thiếu file hoặc chứa file ngoài danh sách được phép.')
        if sum(i.file_size for i in bundle.infolist()) > MAX_INPUT_BYTES:
            raise ValueError('Nội dung ZIP quá lớn.')
        raw = {name:bundle.read(PREFIX+name) for name in INPUT_FILES}
else:
    if not (source/'manifest.json').is_file() and (source/PREFIX/'manifest.json').is_file():
        source = source/PREFIX
    if not source.is_dir() or {p.relative_to(source).as_posix() for p in source.rglob('*') if p.is_file()} != set(INPUT_FILES):
        raise ValueError('Thư mục input thiếu file hoặc chứa file ngoài danh sách được phép.')
    if sum((source/name).stat().st_size for name in INPUT_FILES) > MAX_INPUT_BYTES:
        raise ValueError('Nội dung input quá lớn.')
    raw = {name:(source/name).read_bytes() for name in INPUT_FILES}
manifest = json.loads(raw['manifest.json'])
if (manifest.get('schema_version') != '1.0.0' or manifest.get('encoder') != EXPECTED_ENCODER
        or manifest.get('runtime_loader') != 'transformers-fp16-balanced-v2'):
    raise ValueError('Input Qwen cũ/sai cấu hình. Upload ZIP mới bản sửa OOM v2 và chọn đúng Dataset version.')
if manifest.get('sha256') != {name:hashlib.sha256(value).hexdigest() for name,value in raw.items() if name!='manifest.json'}:
    raise ValueError('Input SHA256 không khớp.')
# Chỉ dùng code sau khi kiểm tra danh sách file và hashes của gói do dự án tạo.
WORK_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR = WORK_ROOT/('qwen3-4b-'+uuid.uuid4().hex)
WORK_DIR.mkdir()
for name,value in raw.items():
    target = WORK_DIR/name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
print('Input:', source, '| Working:', WORK_DIR)


Input: /kaggle/input/datasets/tintthn/pilot-retrieval-qwen3-4b-input/pilot-retrieval-qwen3-4b | Working: /kaggle/working/qwen3-4b-9492e293df684f0eb1ab45d2c1c195f9


## Cài thư viện riêng
Giữ PyTorch CUDA có sẵn của Kaggle. Transformers được pin phiên bản; bản này không dùng SentenceTransformer để tải model, tránh bước `.to(cuda:0)` gom model lại vào một GPU. Cần session mới và Internet.


In [2]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                       str(WORK_DIR/'research/retrieval/qwen3_4b/requirements.txt')])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


0

In [3]:
sys.path.insert(0, str(WORK_DIR/'backend'))
sys.path.insert(0, str(WORK_DIR))
import torch, time, platform, importlib.metadata
from research.retrieval.qwen3_4b.experiment import (
    ENCODER, load_model, encode_texts, build_index, query_document,
    evaluate, comparison, export_artifact, json_bytes, ARTIFACT_FILES, OUTPUT_NAME,
    LOADER_VERSION, gpu_memory, gpu_sync, reset_peak_memory, peak_memory)
from app.retrieval import embedding_text, corpus_hash, digest
from research.retrieval.evaluate import load_inputs

if not torch.cuda.is_available():
    raise RuntimeError('Kaggle Settings → Accelerator → GPU, rồi Save & Run All.')
chunks, dataset = load_inputs(WORK_DIR)
if ENCODER != EXPECTED_ENCODER or manifest['corpus_sha256'] != corpus_hash(chunks) or manifest['query_set_sha256'] != digest(dataset):
    raise ValueError('Model/corpus/query không khớp manifest.')
environment = {'runtime_platform':'kaggle', 'python':platform.python_version(),
               'torch':torch.__version__, 'cuda':torch.version.cuda,
               'gpu':torch.cuda.get_device_name(0), 'gpu_count_available':torch.cuda.device_count(),
               'runtime_loader':LOADER_VERSION, 'precision':'float16', 'attention':'sdpa', 'batch_size':2,
               'packages':{name:importlib.metadata.version(name) for name in
                           ['transformers','pydantic','accelerate']}}
print(json.dumps({'chunks':len(chunks),'queries':len(dataset['queries']),
                  'encoder':ENCODER,'environment':environment},ensure_ascii=False,indent=2))

print("VRAM trước khi tải:", json.dumps(gpu_memory(), ensure_ascii=False, indent=2))


{
  "chunks": 40,
  "queries": 90,
  "encoder": {
    "model_id": "Qwen/Qwen3-Embedding-4B",
    "revision": "5cf2132abc99cad020ac570b19d031efec650f2b",
    "dimensions": 2560,
    "max_length": 512,
    "pooling": "last_token",
    "normalize": true,
    "query_prefix": "Instruct: Given a Vietnamese math learning question, retrieve relevant explanations or hints addressing the mathematical concept and the requested level of support.\nQuery:",
    "passage_prefix": ""
  },
  "environment": {
    "runtime_platform": "kaggle",
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "cuda": "12.8",
    "gpu": "Tesla T4",
    "gpu_count_available": 2,
    "runtime_loader": "transformers-fp16-balanced-v2",
    "precision": "float16",
    "attention": "sdpa",
    "batch_size": 2,
    "packages": {
      "transformers": "4.57.6",
      "pydantic": "2.13.5",
      "accelerate": "1.14.0"
    }
  }
}
VRAM trước khi tải: [
  {
    "device": 0,
    "name": "Tesla T4",
    "free_bytes": 155260682

## Tải model và tạo embedding — sửa OOM v2
Tải bằng AutoModel với `dtype=torch.float16` và Accelerate. T4 ×2 dùng `device_map="balanced"`; một GPU dùng GPU còn đủ VRAM. Chừa 2 GiB mỗi GPU, kiểm tra dtype thật sau tải, in device map. Không gọi `.to(cuda:0)` lên model đã chia GPU; không tự quantize/CPU offload.

Nếu báo thiếu VRAM, dừng session cũ và mở session GPU mới, rồi Run All. Batch 2 chỉ áp dụng khi encode; encode OOM sẽ thử batch 1. Không cắt input quá 512 token. Cache weights ở `/kaggle/temp`, không xuất vào Output.


In [4]:
if 'model' in globals():
    raise RuntimeError('Model đã tồn tại trong session. Restart session trước khi Run All.')
started = time.perf_counter()
reset_peak_memory()
model = load_model(CACHE_ROOT/'qwen3-4b-model-cache')
gpu_sync()
timings = {'model_load_seconds':round(time.perf_counter()-started,3)}
environment.update(model.runtime)
print('Model đã tải:', ENCODER['model_id'], ENCODER['revision'])
print(json.dumps(model.runtime,ensure_ascii=False,indent=2))


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model đã tải: Qwen/Qwen3-Embedding-4B 5cf2132abc99cad020ac570b19d031efec650f2b
{
  "runtime_loader": "transformers-fp16-balanced-v2",
  "device_map": {
    "embed_tokens": "0",
    "layers.0": "0",
    "layers.1": "0",
    "layers.2": "0",
    "layers.3": "0",
    "layers.4": "0",
    "layers.5": "0",
    "layers.6": "0",
    "layers.7": "0",
    "layers.8": "0",
    "layers.9": "0",
    "layers.10": "0",
    "layers.11": "0",
    "layers.12": "0",
    "layers.13": "0",
    "layers.14": "0",
    "layers.15": "0",
    "layers.16": "1",
    "layers.17": "1",
    "layers.18": "1",
    "layers.19": "1",
    "layers.20": "1",
    "layers.21": "1",
    "layers.22": "1",
    "layers.23": "1",
    "layers.24": "1",
    "layers.25": "1",
    "layers.26": "1",
    "layers.27": "1",
    "layers.28": "1",
    "layers.29": "1",
    "layers.30": "1",
    "layers.31": "1",
    "layers.32": "1",
    "layers.33": "1",
    "layers.34": "1",
    "layers.35": "1",
    "norm": "1",
    "rotary_emb": "1"
  

In [7]:
BATCH_SIZE = 2
if not 1 <= BATCH_SIZE <= 8:
    raise ValueError('BATCH_SIZE phải trong khoảng 1–8.')
environment['batch_size'] = BATCH_SIZE

def timed_encode(texts, name, *, queries=False):
    batch = BATCH_SIZE
    started = time.perf_counter()
    while True:
        try:
            values = encode_texts(model,texts,queries=queries,batch_size=batch)
            break
        except torch.cuda.OutOfMemoryError:
            if batch == 1:
                raise
            batch = max(1,batch//2)
            torch.cuda.empty_cache()
            print('Giảm batch do hết VRAM:', batch)
    gpu_sync()
    timings[name] = round(time.perf_counter()-started,3)
    timings[name+'_batch_size'] = batch
    return values

corpus_vectors = timed_encode([embedding_text(c) for c in chunks],'corpus_encode_seconds')
query_vectors = timed_encode([q['request']['query'] for q in dataset['queries']],
                             'query_encode_seconds',queries=True)
timings['peak_gpu_memory_mib_by_device'] = peak_memory()
timings['peak_gpu_memory_mib'] = sum(timings['peak_gpu_memory_mib_by_device'].values())
def write_json(name,value):
    (WORK_DIR/name).write_bytes(json_bytes(value))
write_json('index.json',build_index(chunks,corpus_vectors,environment))
write_json('query-vectors.json',query_document(dataset['queries'],query_vectors))
print(json.dumps(timings,indent=2))


{
  "model_load_seconds": 78.94,
  "corpus_encode_seconds": 1.521,
  "corpus_encode_seconds_batch_size": 2,
  "query_encode_seconds": 3.368,
  "query_encode_seconds_batch_size": 2,
  "peak_gpu_memory_mib_by_device": {
    "0": 3841.2,
    "1": 3870.8
  },
  "peak_gpu_memory_mib": 7712.0
}


## Đánh giá và so sánh với BGE-M3
Dùng đúng bộ lọc concept/purpose/hint level, K=3 và nhãn relevance của bản BGE. Không fallback semantic. Tính cả Recall@3, Hit@3, nDCG@3, từng concept và từng query; không kết luận model lớn hơn tốt hơn chỉ từ một chỉ số. 80 query dương; 10 case còn lại kiểm tra scope không có nội dung.


In [8]:
documents = {name:json.loads((WORK_DIR/name).read_text(encoding='utf-8'))
             for name in ARTIFACT_FILES if name!='evaluation.json'}
report = evaluate(documents)
write_json('evaluation.json',report)
scores = comparison(documents,report)
for method,value in scores.items():
    print(f"{method:22} Recall@3={value['recall_at_3']:.4f}  Hit@3={value['hit_at_3']:.4f}  nDCG@3={value['ndcg_at_3']:.4f}")
print('Chi tiết từng query/concept trong evaluation.json. Nhãn dev chưa được review.')


metadata               Recall@3=0.6062  Hit@3=0.6125  nDCG@3=0.4885
BM25                   Recall@3=0.8396  Hit@3=0.9625  nDCG@3=0.8517
BGE-M3                 Recall@3=0.8750  Hit@3=0.9375  nDCG@3=0.8544
Qwen3-Embedding-4B     Recall@3=0.9583  Hit@3=1.0000  nDCG@3=0.9544
Chi tiết từng query/concept trong evaluation.json. Nhãn dev chưa được review.


## Xuất Output
Chỉ xuất JSON; không chứa model weights hoặc code. File ở ngay gốc `/kaggle/working/pilot-retrieval-qwen3-4b.zip`. Tải trong Output của phiên Save & Run All đã hoàn thành, đặt vào `data/exports/` tại máy. Không dùng công cụ import BGE cho file này.


In [9]:
output_zip = export_artifact(WORK_DIR,WORK_ROOT/OUTPUT_NAME,environment,timings,
                             hashlib.sha256(raw['manifest.json']).hexdigest())
print('Hoàn thành. Tải file này trong Kaggle Output:',output_zip)


Hoàn thành. Tải file này trong Kaggle Output: /kaggle/working/pilot-retrieval-qwen3-4b.zip


## Kiểm tra/nhập riêng khi chạy xong
Trong dự án, dùng công cụ riêng:
```powershell
.\.venv\Scripts\python.exe research/retrieval/qwen3_4b/experiment.py import data/exports/pilot-retrieval-qwen3-4b.zip
```
Kết quả chỉ nhập vào `data/retrieval/qwen3-4b-v1/`. Công cụ kiểm tra hashes/model/corpus/query/vector và tính lại rankings trước khi ghi. Không thay `.env`, backend, index BGE hoặc nội dung. Backend hiện tại không tự dùng Qwen; lựa chọn model sản phẩm chỉ quyết định sau so sánh.

Nguồn cấu hình: [Qwen model card](https://huggingface.co/Qwen/Qwen3-Embedding-4B), [pooling](https://huggingface.co/Qwen/Qwen3-Embedding-4B/blob/5cf2132abc99cad020ac570b19d031efec650f2b/1_Pooling/config.json), [Accelerate](https://huggingface.co/docs/accelerate/en/concept_guides/big_model_inference), [Kaggle](https://www.kaggle.com/docs/notebooks).
